# ema-first-moment — faded example 1: Fill the first-moment recurrence

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-first-moment`. The last cell reports your progress on the `Optimizer: Adam EMA first moment` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA first moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-first-moment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-first-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA first moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The Adam first moment is `m = beta1*m + (1-beta1)*g`. The `(1-beta1)` weight on the fresh gradient is the defining feature versus plain momentum. The buffer must be mutated in place with `copy_` so its storage identity is preserved.

## Faded exercise 1

Implement `ema_m_step(m, g, beta1)`. Update the first-moment buffer `m` in place using the Adam recurrence and return it. Complete the blanked in-place update.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(3)
beta1 = 0.9
m = t.zeros(3)
g = t.tensor([2.0, -1.0, 4.0])

def ema_m_step(m, g, beta1):
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    return m

m2 = t.zeros(3)
ema_m_step(m2, g, beta1)
print(m2.tolist())


def _test():
    b1 = 0.9
    # two-step independent ground truth
    g1 = t.tensor([2.0, -1.0, 4.0])
    g2 = t.tensor([0.0, 3.0, -2.0])
    buf = t.zeros(3)
    ptr = buf.data_ptr()
    ema_m_step(buf, g1, b1)
    ema_m_step(buf, g2, b1)
    # closed form: m2 = (1-b1)*(b1*g1 + g2)
    expected = (1 - b1) * (b1 * g1 + g2)
    assert t.allclose(buf, expected, atol=1e-6), (buf, expected)
    # in-place: storage identity preserved
    assert buf.data_ptr() == ptr
    # sign preserved (unlike second moment) — entry 2 stays positive after g1>0 dominates? check entry 0
    assert buf[0] > 0


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(3)
beta1 = 0.9
m = t.zeros(3)
g = t.tensor([2.0, -1.0, 4.0])

def ema_m_step(m, g, beta1):
    m.copy_(beta1 * m + (1 - beta1) * g)
    return m

m2 = t.zeros(3)
ema_m_step(m2, g, beta1)
print(m2.tolist())
```
</details>